In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("bethesda-classification")

### Служебная информация об эксперименте

В этой ячейке фиксируется служебная информация для воспроизводимости эксперимента: имя участника, название задачи, дата запуска и SHA текущего Git-коммита.

In [3]:
import subprocess
from datetime import datetime

# Основная информация об эксперименте
participant = "Savely"
task_name = "bethesda-classification"

# Автоматически получаем SHA текущего Git-коммита
git_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True
).strip()

# Сохраняем текущую дату запуска эксперимента
run_date = datetime.now().strftime("%Y-%m-%d")

print("Participant:", participant)
print("Task:", task_name)
print("Date:", run_date)
print("Git commit:", git_commit)

Participant: Savely
Task: bethesda-classification
Date: 2026-10-10
Git commit: 5a6f839b2864ebff15057532898f031059e50895


### Версия датасета

Для воспроизводимости экспериментов используется SHA256-хеш файла `dataset_first.csv`.

Логика вычисления хеша вынесена в общий модуль `src/common/dataset_versioning.py`,
чтобы одна и та же реализация использовалась при архивировании датасетов и в экспериментах классификации.

При сохранении версии датасета отдельная копия помещается в каталог
`data/processed/versions`, а первые 12 символов SHA256 добавляются к имени файла.

В MLflow будет сохраняться полный SHA256-хеш, что позволяет однозначно связать
каждый запуск модели с конкретной версией данных.

In [10]:
from pathlib import Path

from src.common.dataset_versioning import calculate_sha256

# Определяем корень проекта
PROJECT_ROOT = Path.cwd()

# Если Jupyter запущен из папки notebooks/classification,
# поднимаемся до корня проекта
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parents[1]

# Путь к основному датасету классификации
dataset_path = PROJECT_ROOT / "data" / "processed" / "dataset_first.csv"

# Вычисляем SHA256 текущей версии датасета
dataset_hash = calculate_sha256(dataset_path)

# Короткая версия используется только для удобного отображения
dataset_version = dataset_hash[:12]

print("Dataset:", dataset_path.name)
print("Dataset version:", dataset_version)
print("Full SHA256:", dataset_hash)

Dataset: dataset_first.csv
Dataset version: b5825a9b4570
Full SHA256: b5825a9b457023d0c180c5ebc2ad081d183bd30c4ea1d593031522c4e50730e9


 ## Загрузка и первичная проверка датасета

Загрузим текущую версию `dataset_first.csv`, для которой ранее был вычислен SHA256-хеш.

На этом этапе проверим размер датасета, названия столбцов и первые строки данных.
Это позволяет убедиться, что файл корректно загружен и готов к дальнейшему анализу.

Модель пока не обучается, поэтому отдельный запуск MLflow на этом этапе не создаётся.

In [ ]:
import pandas as pd

# Проверяем, что файл датасета существует
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Датасет не найден: {dataset_path}"
    )

# Загружаем подготовленный датасет
df = pd.read_csv(dataset_path, sep=";")

print("Путь к датасету:", dataset_path)
print("Размер датасета:", df.shape)
print("Столбцы:", df.columns.tolist())

df.head()

## Проверка структуры и качества данных

Проверим названия столбцов, количество пропущенных значений
и распределение объектов по классам Bethesda.
Это необходимо перед разделением данных на обучающую и тестовую выборки.

In [13]:
print("Столбцы:")
print(df.columns.tolist())

print("\nПропущенные значения:")
print(df.isna().sum())

print("\nКоличество объектов каждого класса Bethesda:")
print(df["bethesda"].value_counts().sort_index())

print("\nДоля каждого класса:")
print(df["bethesda"].value_counts(normalize=True).sort_index().round(4))

Столбцы:
['case_id', 'description', 'conclusion', 'bethesda']

Пропущенные значения:
case_id        0
description    0
conclusion     0
bethesda       0
dtype: int64

Количество объектов каждого класса Bethesda:
bethesda
I       576
II     3814
III     287
IV      549
V       166
VI      195
Name: count, dtype: int64

Доля каждого класса:
bethesda
I      0.1031
II     0.6827
III    0.0514
IV     0.0983
V      0.0297
VI     0.0349
Name: proportion, dtype: float64


Всего 5587 строк, пропусков нет, целевая переменная заполнена полностью. Но классы заметно несбалансированы: Bethesda II занимает 68.27%, а самые редкие V и VI - около 3% каждая.

## Проверка дубликатов

Перед разделением данных на обучающую и тестовую выборки проверим наличие
повторяющихся идентификаторов и текстов `description`.

Особенно важно исключить ситуацию, когда одинаковые описания попадают
одновременно в train и test, поскольку это может привести к утечке данных
и завышенной оценке качества модели.

Поле `conclusion` не используется как вход модели. Для классификации
используется только `description`, а `bethesda` является целевой переменной.

In [14]:
# Проверяем уникальность идентификаторов случаев
duplicate_case_ids = df["case_id"].duplicated().sum()

# Проверяем полные дубликаты строк
duplicate_rows = df.duplicated().sum()

# Проверяем повторяющиеся тексты description
duplicate_descriptions = df["description"].duplicated().sum()

print("Дубликаты case_id:", duplicate_case_ids)
print("Полные дубликаты строк:", duplicate_rows)
print("Повторяющиеся description:", duplicate_descriptions)

# Проверяем, встречается ли один и тот же description
# с разными категориями Bethesda
description_bethesda_counts = (
    df.groupby("description")["bethesda"]
    .nunique()
)

conflicting_descriptions = description_bethesda_counts[
    description_bethesda_counts > 1
]

print(
    "Description с разными категориями Bethesda:",
    len(conflicting_descriptions)
)

Дубликаты case_id: 0
Полные дубликаты строк: 0
Повторяющиеся description: 0
Description с разными категориями Bethesda: 0


## Разделение данных на обучающую и тестовую выборки

В качестве входного признака модели используется только поле `description`,
а целевой переменной является категория `bethesda`.

Данные разделяются на обучающую и тестовую выборки в отношении 80/20.
Используется стратифицированное разделение, чтобы сохранить исходное
соотношение классов Bethesda в обеих выборках.

Параметр `random_state=42` фиксирует случайное разделение и делает эксперимент
воспроизводимым.

TF-IDF будет обучаться только на обучающей выборке после разделения данных,
чтобы исключить утечку информации из test в train.

In [15]:
from sklearn.model_selection import train_test_split

# В качестве входа модели используем только текст description
X = df["description"]

# Целевая переменная - категория Bethesda
y = df["bethesda"]

# Делим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Размер всей выборки:", len(X))
print("Размер train:", len(X_train))
print("Размер test:", len(X_test))

print("\nРаспределение классов в train:")
print(y_train.value_counts().sort_index())

print("\nРаспределение классов в test:")
print(y_test.value_counts().sort_index())

Размер всей выборки: 5587
Размер train: 4469
Размер test: 1118

Распределение классов в train:
bethesda
I       461
II     3051
III     229
IV      439
V       133
VI      156
Name: count, dtype: int64

Распределение классов в test:
bethesda
I      115
II     763
III     58
IV     110
V       33
VI      39
Name: count, dtype: int64
